<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/study_design/cohort_construction/Final_Cohort_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 1: Setup & Library Imports
# ============================================================
# Notebook purpose: COHORT CONSTRUCTION ONLY
#   - Observation periods
#   - Disease patient identification
#   - Index date assignment
#   - Eligibility filtering
#   - Output: clean cohort CSV ready for analysis notebooks
#
# Run this cell first every time you open the notebook.
# No changes needed between diseases.
# ============================================================

# -- Install required libraries --
!pip install openpyxl --quiet    # Excel read/write for output tables

# -- Standard library imports --
import os
import re
import warnings
from datetime import datetime

# -- Data manipulation --
import pandas as pd
import numpy as np

# -- Suppress non-critical warnings --
warnings.filterwarnings("ignore")
pd.set_option("future.no_silent_downcasting", True)

# -- Global utility: strip timezone from any datetime series --
def to_naive(series):
    """Parse to datetime and strip timezone if present.
    Needed because Synthea mixes tz-naive and tz-aware timestamps.
    """
    s = pd.to_datetime(series, errors="coerce")
    if hasattr(s, "dt") and s.dt.tz is not None:
        s = s.dt.tz_localize(None)
    return s

# -- Confirmation --
print("=" * 55)
print("  MOSAIC -- Cohort Construction Notebook")
print("  Cell 1: Libraries loaded successfully")
print("=" * 55)
print(f"  pandas   {pd.__version__}")
print(f"  numpy    {np.__version__}")
print("=" * 55)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 2: Paths, Parameters & Disease Configuration
# ============================================================
# THIS IS THE ONLY CELL YOU CHANGE BETWEEN DISEASES:
#   1. DISEASE_NAME
#   2. CONDITION_SEARCH_KEYWORDS
#   3. TREATMENT_SEARCH_KEYWORDS
#   4. EXCLUDE_KEYWORDS (if needed)
# Everything else is automatic.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# PATHS
# ============================================================

BASE_PATH        = "/content/drive/MyDrive/THESIS/Data/Coherent_data/csv/"
UNIQUE_DESC_PATH = "/content/drive/MyDrive/THESIS/Data/Coherent_data/unique_descriptions_coherent_dataset.csv"
OUTPUT_PATH      = "/content/drive/MyDrive/THESIS/Data/Coherent_data/cohort_outputs/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# ============================================================
# AUTO-DETECT STUDY PERIOD
# Scans ALL clinical CSV files to find the earliest and
# latest date in the entire dataset.
#
# DATASET_START_DATE → first ever recorded clinical event
# DATASET_END_DATE   → last ever recorded clinical event
#
# DATASET_END_DATE is used (not patient OBS_END) for the
# follow-up eligibility check to avoid survivorship bias:
# a patient who dies 2 years after their index date should
# still be eligible — their death IS the outcome we study.
#
# Eligibility rule:
#   DATASET_END_DATE - index_date >= FOLLOWUP_YEARS
#
# Time-to-event (computed in analysis notebook):
#   duration = min(DEATH_DATE, index_date + 5yr) - index_date
#   event    = 1 if died within 5yr, 0 if censored
# ============================================================

FILES_TO_SCAN = [
    ("conditions.csv",       ["START", "STOP"]),
    ("observations.csv",     ["DATE"]),
    ("medications.csv",      ["START", "STOP"]),
    ("encounters.csv",       ["START", "STOP"]),
    ("procedures.csv",       ["DATE"]),
    ("careplans.csv",        ["START", "STOP"]),
    ("immunizations.csv",    ["DATE"]),
    ("imaging_studies.csv",  ["DATE"]),
    ("devices.csv",          ["START", "STOP"]),
    ("allergies.csv",        ["START", "STOP"]),
    ("supplies.csv",         ["DATE"]),
    ("patients.csv",         ["BIRTHDATE", "DEATHDATE"]),
]

print("Scanning all CSV files to detect study period...")
print("-" * 55)

_all_min_dates = []
_all_max_dates = []

for filename, date_cols in FILES_TO_SCAN:
    filepath = os.path.join(BASE_PATH, filename)
    if not os.path.exists(filepath):
        print(f"  MISSING: {filename} — skipping")
        continue

    _df = pd.read_csv(filepath, low_memory=False)
    _file_dates = []

    for col in date_cols:
        if col not in _df.columns:
            continue
        _parsed = to_naive(_df[col])
        _valid  = _parsed.dropna()
        if len(_valid) > 0:
            _file_dates.extend(_valid.tolist())

    if _file_dates:
        _file_min = min(_file_dates)
        _file_max = max(_file_dates)
        _all_min_dates.append(_file_min)
        _all_max_dates.append(_file_max)
        print(f"  {filename:<30} {str(_file_min.date()):<14} → {_file_max.date()}")

    del _df

print("-" * 55)

DATASET_START_DATE = min(_all_min_dates)
DATASET_END_DATE   = max(_all_max_dates)

print(f"\n  STUDY PERIOD: {DATASET_START_DATE.date()}  →  {DATASET_END_DATE.date()}")
print(f"  Total span  : {(DATASET_END_DATE - DATASET_START_DATE).days / 365.25:.1f} years")

del _all_min_dates, _all_max_dates

# ============================================================
# GLOBAL PARAMETERS — same for all diseases
# ============================================================

LOOKBACK_YEARS  = 5    # years of data required BEFORE index date
FOLLOWUP_YEARS  = 5    # years of dataset coverage required AFTER index date
RANDOM_SEED     = 42   # for reproducible random date assignment

# ============================================================
# DISEASE CONFIGURATION — CHANGE THIS BLOCK PER DISEASE
# ============================================================

DISEASE_NAME = "T2D"

# Keywords to find CONDITION codes in the unique descriptions catalog
CONDITION_SEARCH_KEYWORDS = [
    "type 2 diabetes",
    "diabetes mellitus type 2",
    "neuropathy due to type 2",
    "retinopathy due to type 2",
    "due to type 2 diabetes",
    "macular edema and retinopathy due to type 2",
    "microalbuminuria due to type 2",
    "proteinuria due to type 2",
    "blindness due to type 2",
    "44054006",
]

# Keywords to find TREATMENT codes in the unique descriptions catalog
TREATMENT_SEARCH_KEYWORDS = [
    "metformin",
    "insulin",
    "liraglutide",
    "semaglutide",
    "dulaglutide",
    "exenatide",
    "empagliflozin",
    "dapagliflozin",
    "canagliflozin",
    "ertugliflozin",
    "sitagliptin",
    "saxagliptin",
    "alogliptin",
    "linagliptin",
    "glipizide",
    "glyburide",
    "glimepiride",
    "pioglitazone",
    "rosiglitazone",
    "repaglinide",
    "nateglinide",
    "acarbose",
    "pramlintide",
]

# Terms to EXCLUDE from condition results after keyword search
EXCLUDE_KEYWORDS = [
    "cystic fibrosis",
    "type 1",
    "gestational",
    "prediabetes",
]

# ============================================================
# CONFIRMATION
# ============================================================

print()
print("=" * 55)
print(f"  MOSAIC -- Cohort Construction: {DISEASE_NAME}")
print("=" * 55)
print(f"  Study period   : {DATASET_START_DATE.date()} → {DATASET_END_DATE.date()}")
print(f"  Lookback yrs   : {LOOKBACK_YEARS}")
print(f"  Follow-up yrs  : {FOLLOWUP_YEARS}")
print(f"  Random seed    : {RANDOM_SEED}")
print(f"  Output path    : {OUTPUT_PATH}")
print(f"  Condition kws  : {len(CONDITION_SEARCH_KEYWORDS)} keywords")
print(f"  Treatment kws  : {len(TREATMENT_SEARCH_KEYWORDS)} keywords")
print(f"  Exclude kws    : {len(EXCLUDE_KEYWORDS)} keywords")
print("=" * 55)
print("  Ready — run Cell 3 to extract codes from catalog")
print("=" * 55)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 3: Catalog Extractor
# ============================================================
# Reads the unique descriptions catalog and extracts:
#   - CONDITION_DESCRIPTIONS / CONDITION_CODE_LIST
#     for identifying disease patients in conditions.csv
#   - TREATMENT_DESCRIPTIONS / TREATMENT_CODE_LIST
#     for identifying first treatment in medications.csv
#   - CONDITION_SOURCE_MAP
#     for finding earliest evidence dates across all files
#
# IMPORTANT DESIGN DECISION:
#   Patient identification (WHO has the disease) uses
#   conditions.csv ONLY — via CONDITION_DESCRIPTIONS and
#   CONDITION_CODE_LIST.
#
#   CONDITION_SOURCE_MAP is used ONLY to find earliest
#   date for already-confirmed patients across other files.
#   It never adds new patients.
#
# No changes needed between diseases — update keywords
# in Cell 2 only.
# ============================================================

# ============================================================
# STEP 1: Load and parse catalog
# ============================================================

print("Loading unique descriptions catalog...")

catalog = pd.read_csv(UNIQUE_DESC_PATH, low_memory=False)
print(f"  Total unique descriptions : {len(catalog):,}")
print(f"  Source files present      : "
      f"{sorted(catalog['Source_Files'].unique())}")

def parse_catalog_entry(raw):
    """Split catalog Description into clean_desc, code, reason_desc."""
    if " (code:" in raw:
        clean_desc = raw.split(" (code:")[0].strip()
        inside     = raw.split(" (code:")[1].rstrip(")")
        code_part  = inside.split(",")[0].strip()
        reason_desc = (
            inside.split("REASONDESCRIPTION:")[1].strip().rstrip(")")
            if "REASONDESCRIPTION:" in inside else None
        )
    else:
        clean_desc  = raw.strip()
        code_part   = None
        reason_desc = None
    return {
        "clean_desc":  clean_desc,
        "code":        code_part,
        "reason_desc": reason_desc,
    }

parsed             = catalog["Description"].apply(parse_catalog_entry)
catalog["CLEAN_DESC"]  = parsed.apply(lambda x: x["clean_desc"])
catalog["CODE"]        = parsed.apply(lambda x: x["code"])
catalog["REASON_DESC"] = parsed.apply(lambda x: x["reason_desc"])

print(f"  Catalog parsed successfully")

# ============================================================
# SOURCE FILE CONFIG
# ============================================================

SOURCE_CONFIG = {
    "conditions":   {
        "csv":        "conditions.csv",
        "date_col":   "START",
        "patient_col":"PATIENT",
        "match_cols": ["DESCRIPTION"],
    },
    "medications":  {
        "csv":        "medications.csv",
        "date_col":   "START",
        "patient_col":"PATIENT",
        "match_cols": ["DESCRIPTION"],
    },
    "encounters":   {
        "csv":        "encounters.csv",
        "date_col":   "START",
        "patient_col":"PATIENT",
        "match_cols": ["DESCRIPTION", "REASONDESCRIPTION"],
    },
    "observations": {
        "csv":        "observations.csv",
        "date_col":   "DATE",
        "patient_col":"PATIENT",
        "match_cols": ["DESCRIPTION"],
    },
    "procedures":   {
        "csv":        "procedures.csv",
        "date_col":   "DATE",
        "patient_col":"PATIENT",
        "match_cols": ["DESCRIPTION", "REASONDESCRIPTION"],
    },
    "careplans":    {
        "csv":        "careplans.csv",
        "date_col":   "START",
        "patient_col":"PATIENT",
        "match_cols": ["DESCRIPTION", "REASONDESCRIPTION"],
    },
}

# ============================================================
# STEP 2: Extract CONDITION codes — conditions.csv only
# ============================================================

print("\n" + "="*60)
print(f"CONDITION CODES FOR: {DISEASE_NAME}")
print("="*60)

cond_pattern = "|".join(CONDITION_SEARCH_KEYWORDS)
excl_pattern = "|".join(EXCLUDE_KEYWORDS) if EXCLUDE_KEYWORDS else None

# Search only conditions source entries in catalog
cond_catalog_all = catalog[
    catalog["Source_Files"].str.contains(
        "conditions", case=False, na=False
    )
].copy()

# Match on CLEAN_DESC or REASON_DESC
cond_mask = (
    cond_catalog_all["CLEAN_DESC"].str.contains(
        cond_pattern, case=False, na=False
    ) |
    cond_catalog_all["REASON_DESC"].str.contains(
        cond_pattern, case=False, na=False
    )
)

# Apply exclusions
if excl_pattern:
    excl_mask = (
        cond_catalog_all["CLEAN_DESC"].str.contains(
            excl_pattern, case=False, na=False
        ) |
        cond_catalog_all["REASON_DESC"].str.contains(
            excl_pattern, case=False, na=False
        )
    )
    cond_mask = cond_mask & ~excl_mask

cond_catalog = cond_catalog_all[cond_mask].copy()

# Print what was found
print(f"\n  Found via keyword search:")
for _, row in cond_catalog.iterrows():
    code_str   = f"code: {row['CODE']}" if row["CODE"] else ""
    print(f"    {row['CLEAN_DESC']:<65} {code_str}")

# Build lists — descriptions AND codes for matching
CONDITION_DESCRIPTIONS = cond_catalog["CLEAN_DESC"].unique().tolist()
CONDITION_CODE_LIST    = cond_catalog["CODE"].dropna().unique().tolist()

# ── CRITICAL: also add the primary disease SNOMED codes
# that may not appear in keyword search because their
# description is ambiguous (e.g. "Diabetes" for T2D).
# These must be verified manually and added to
# CONDITION_SEARCH_KEYWORDS as raw code strings.
# ──
# Check if any keyword looks like a SNOMED code
# (all digits) — add directly to code list
for kw in CONDITION_SEARCH_KEYWORDS:
    if kw.strip().isdigit() and kw.strip() not in CONDITION_CODE_LIST:
        CONDITION_CODE_LIST.append(kw.strip())
        # Also find its description in conditions catalog
        code_match = cond_catalog_all[
            cond_catalog_all["CODE"] == kw.strip()
        ]
        if len(code_match) > 0:
            for desc in code_match["CLEAN_DESC"].unique():
                if desc not in CONDITION_DESCRIPTIONS:
                    CONDITION_DESCRIPTIONS.append(desc)
                    print(f"    {desc:<65} code: {kw}  "
                          f"[added via direct code match]")
        else:
            print(f"    [code {kw} not found in catalog — "
                  f"will match by CODE column only]")

print(f"\n  TOTAL condition descriptions : {len(CONDITION_DESCRIPTIONS)}")
print(f"  TOTAL condition codes        : {len(CONDITION_CODE_LIST)}")
print(f"\n  CONDITION_DESCRIPTIONS:")
for d in sorted(CONDITION_DESCRIPTIONS):
    print(f"    '{d}'")
print(f"\n  CONDITION_CODE_LIST:")
for c in sorted(CONDITION_CODE_LIST):
    print(f"    '{c}'")

# ============================================================
# STEP 3: Build CONDITION_SOURCE_MAP
#         For finding earliest dates across other files
#         (confirmed patients only — see Cell 5)
# ============================================================

print(f"\n  Building source map for earliest date search...")

cond_pattern_all = "|".join(CONDITION_SEARCH_KEYWORDS)

CONDITION_SOURCE_MAP = {}

for source, cfg in SOURCE_CONFIG.items():
    source_entries = catalog[
        catalog["Source_Files"].str.contains(
            source, case=False, na=False
        )
    ]
    source_matches = source_entries[
        source_entries["CLEAN_DESC"].str.contains(
            cond_pattern_all, case=False, na=False
        ) |
        source_entries["REASON_DESC"].str.contains(
            cond_pattern_all, case=False, na=False
        )
    ]
    if excl_pattern:
        excl = (
            source_matches["CLEAN_DESC"].str.contains(
                excl_pattern, case=False, na=False
            ) |
            source_matches["REASON_DESC"].str.contains(
                excl_pattern, case=False, na=False
            )
        )
        source_matches = source_matches[~excl]

    if len(source_matches) > 0:
        CONDITION_SOURCE_MAP[source] = {
            "descs":  source_matches["CLEAN_DESC"].tolist(),
            "codes":  source_matches["CODE"].dropna().tolist(),
            "config": cfg,
        }
        print(f"    {source:<15} : {len(source_matches)} descriptions")

# ============================================================
# STEP 4: Extract TREATMENT codes — medications only
# ============================================================

print("\n" + "="*60)
print(f"TREATMENT CODES FOR: {DISEASE_NAME}")
print("="*60)

meds_catalog = catalog[
    catalog["Source_Files"].str.contains(
        "medication", case=False, na=False
    )
].copy()

treat_pattern = "|".join(TREATMENT_SEARCH_KEYWORDS)
treat_mask = (
    meds_catalog["CLEAN_DESC"].str.contains(
        treat_pattern, case=False, na=False
    ) |
    meds_catalog["REASON_DESC"].str.contains(
        treat_pattern, case=False, na=False
    )
)
treat_catalog = meds_catalog[treat_mask].copy()

print(f"\n  [MEDICATIONS] — {len(treat_catalog)} description(s):")
for _, row in treat_catalog.iterrows():
    code_str   = f"code: {row['CODE']}" if row["CODE"] else ""
    reason_str = (f"  reason: {row['REASON_DESC']}"
                  if row["REASON_DESC"] else "")
    print(f"    {row['CLEAN_DESC']:<65} {code_str}{reason_str}")

TREATMENT_DESCRIPTIONS = treat_catalog["CLEAN_DESC"].unique().tolist()
TREATMENT_CODE_LIST    = treat_catalog["CODE"].dropna().unique().tolist()

print(f"\n  TOTAL treatment descriptions : {len(TREATMENT_DESCRIPTIONS)}")
print(f"  TOTAL treatment codes        : {len(TREATMENT_CODE_LIST)}")

# ============================================================
# STEP 5: Review checklist
# ============================================================

print("\n" + "="*60)
print("REVIEW CHECKLIST — verify before proceeding to Cell 4")
print("="*60)
print(f"  Condition descriptions : {len(CONDITION_DESCRIPTIONS)}")
print(f"  Condition codes        : {len(CONDITION_CODE_LIST)}")
print(f"  Treatment descriptions : {len(TREATMENT_DESCRIPTIONS)}")
print(f"  Source map files       : {list(CONDITION_SOURCE_MAP.keys())}")

if len(CONDITION_DESCRIPTIONS) == 0 and len(CONDITION_CODE_LIST) == 0:
    print("  WARNING: No condition codes found — check keywords in Cell 2!")
if len(TREATMENT_DESCRIPTIONS) == 0:
    print("  WARNING: No treatment descriptions found — check keywords!")
print()
print("  If correct → proceed to Cell 4")
print("  If wrong   → adjust keywords in Cell 2 and re-run Cell 3")
print("="*60)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 4: Observation Periods & Death Reconciliation
# ============================================================
# Scans all clinical CSV files to compute per-patient
# observation windows, then reconciles death status from
# three independent sources.
#
# Output variables used by downstream cells:
#   all_patients_df   full patient table with obs periods
#                     and reconciled death information
# ============================================================

# ============================================================
# STEP 1: Load patients.csv for demographics
# ============================================================

print("Loading patients.csv...")

patients = pd.read_csv(
    os.path.join(BASE_PATH, "patients.csv"), low_memory=False
)

# Standardise ID column name
if "Id" in patients.columns:
    patients = patients.rename(columns={"Id": "PATIENT"})
elif "ID" in patients.columns:
    patients = patients.rename(columns={"ID": "PATIENT"})

patients["BIRTHDATE"]  = to_naive(patients["BIRTHDATE"])
patients["DEATHDATE"]  = to_naive(patients["DEATHDATE"])

print(f"  Total patients: {len(patients):,}")
print(f"  Columns kept  : PATIENT, GENDER, RACE, BIRTHDATE, DEATHDATE")

# ============================================================
# STEP 2: Scan all clinical CSVs for observation periods
# ============================================================

print("\nScanning clinical files for observation periods...")
print("-" * 55)

FILES_TO_SCAN = [
    ("conditions.csv",       ["START", "STOP"]),
    ("observations.csv",     ["DATE"]),
    ("medications.csv",      ["START", "STOP"]),
    ("encounters.csv",       ["START", "STOP"]),
    ("procedures.csv",       ["DATE"]),
    ("careplans.csv",        ["START", "STOP"]),
    ("immunizations.csv",    ["DATE"]),
    ("imaging_studies.csv",  ["DATE"]),
    ("devices.csv",          ["START", "STOP"]),
    ("allergies.csv",        ["START", "STOP"]),
    ("supplies.csv",         ["DATE"]),
]

patient_dates = {}  # {patient_id: [list of all dates]}

for filename, date_cols in FILES_TO_SCAN:
    filepath = os.path.join(BASE_PATH, filename)

    if not os.path.exists(filepath):
        print(f"  MISSING : {filename} — skipping")
        continue

    df = pd.read_csv(filepath, low_memory=False)

    if "PATIENT" not in df.columns:
        print(f"  SKIP    : {filename} — no PATIENT column")
        continue

    dates_found = 0
    for col in date_cols:
        if col not in df.columns:
            continue
        parsed     = to_naive(df[col])
        valid_mask = parsed.notna()
        if valid_mask.sum() == 0:
            continue
        dates_found += valid_mask.sum()
        for pid, date in zip(
            df.loc[valid_mask, "PATIENT"], parsed[valid_mask]
        ):
            if pid not in patient_dates:
                patient_dates[pid] = []
            patient_dates[pid].append(date)

    print(f"  OK  {filename:<30} {dates_found:>8,} valid dates")
    del df

print("-" * 55)
print(f"  Patients with at least one clinical date: {len(patient_dates):,}")

# ============================================================
# STEP 3: Compute OBS_START and OBS_END per patient
# ============================================================

print("\nComputing observation windows...")

obs_records = []
for pid, dates in patient_dates.items():
    obs_records.append({
        "PATIENT":           pid,
        "OBS_START":         min(dates),
        "OBS_END":           max(dates),
    })

obs_df = pd.DataFrame(obs_records)
obs_df["OBS_DURATION_YEARS"] = (
    (obs_df["OBS_END"] - obs_df["OBS_START"]).dt.days / 365.25
).round(2)

print(f"  Observation windows computed: {len(obs_df):,} patients")

# ============================================================
# STEP 4: Death reconciliation — 3 sources, priority order
#
# Priority 1: encounters.csv  SNOMED 308646001 (Death Cert)
#             → richest source: has REASONDESCRIPTION = cause
# Priority 2: observations.csv LOINC 69453-9 (Cause of Death)
#             → has cause in VALUE column
# Priority 3: patients.csv DEATHDATE
#             → binary death flag, no cause info
# ============================================================

print("\nReconciling death status from 3 sources...")

# -- Source 1: encounters.csv --
enc = pd.read_csv(
    os.path.join(BASE_PATH, "encounters.csv"), low_memory=False
)
death_enc = enc[
    enc["CODE"].astype(str) == "308646001"
].copy()
death_enc["DEATH_DATE_ENC"] = to_naive(death_enc["START"])

death_enc_dedup = (
    death_enc.sort_values("DEATH_DATE_ENC")
    .groupby("PATIENT")
    .agg(
        DEATH_DATE_ENC     = ("DEATH_DATE_ENC",    "first"),
        CAUSE_OF_DEATH_ENC = ("REASONDESCRIPTION", "first"),
    )
    .reset_index()
)
print(f"  Source 1 (encounters SNOMED 308646001) : "
      f"{death_enc_dedup['PATIENT'].nunique():,} patients")
del enc, death_enc

# -- Source 2: observations.csv --
obs_file = pd.read_csv(
    os.path.join(BASE_PATH, "observations.csv"), low_memory=False
)
death_obs = obs_file[
    obs_file["CODE"].astype(str) == "69453-9"
].copy()
death_obs["DEATH_DATE_OBS"] = to_naive(death_obs["DATE"])

death_obs_dedup = (
    death_obs.sort_values("DEATH_DATE_OBS")
    .groupby("PATIENT")
    .agg(
        DEATH_DATE_OBS     = ("DEATH_DATE_OBS", "first"),
        CAUSE_OF_DEATH_OBS = ("VALUE",          "first"),
    )
    .reset_index()
)
print(f"  Source 2 (observations LOINC 69453-9)  : "
      f"{death_obs_dedup['PATIENT'].nunique():,} patients")
del obs_file, death_obs

# -- Source 3: patients.csv DEATHDATE (already loaded) --
death_pts = patients[patients["DEATHDATE"].notna()][[
    "PATIENT", "DEATHDATE"
]].copy().rename(columns={"DEATHDATE": "DEATH_DATE_PTS"})
print(f"  Source 3 (patients.csv DEATHDATE)       : "
      f"{len(death_pts):,} patients")

# ============================================================
# STEP 5: Merge all sources into one patient table
# ============================================================

print("\nMerging all sources...")

all_patients_df = obs_df.merge(
    patients[["PATIENT", "BIRTHDATE", "DEATHDATE",
              "GENDER", "RACE"]],
    on="PATIENT", how="left"
)
all_patients_df = all_patients_df.merge(
    death_enc_dedup, on="PATIENT", how="left"
)
all_patients_df = all_patients_df.merge(
    death_obs_dedup, on="PATIENT", how="left"
)
all_patients_df = all_patients_df.merge(
    death_pts, on="PATIENT", how="left"
)

# DIED = any source records death
all_patients_df["DIED"] = (
    all_patients_df["DEATH_DATE_ENC"].notna() |
    all_patients_df["DEATH_DATE_OBS"].notna() |
    all_patients_df["DEATH_DATE_PTS"].notna()
)

# Best death date: enc > obs > patients.csv
all_patients_df["FINAL_DEATH_DATE"] = (
    all_patients_df["DEATH_DATE_ENC"]
    .combine_first(all_patients_df["DEATH_DATE_OBS"])
    .combine_first(all_patients_df["DEATH_DATE_PTS"])
)

# Best cause of death: enc > obs
all_patients_df["FINAL_CAUSE_OF_DEATH"] = (
    all_patients_df["CAUSE_OF_DEATH_ENC"]
    .combine_first(all_patients_df["CAUSE_OF_DEATH_OBS"])
)

# Strip tz from all datetime columns
for col in all_patients_df.select_dtypes(
    include=["datetimetz"]
).columns:
    all_patients_df[col] = all_patients_df[col].dt.tz_localize(None)

# ============================================================
# STEP 6: Cross-source agreement check
# ============================================================

print("\nDeath source overlap:")
enc_only  = (
    all_patients_df["DEATH_DATE_ENC"].notna() &
    all_patients_df["DEATH_DATE_OBS"].isna()  &
    all_patients_df["DEATH_DATE_PTS"].isna()
).sum()
obs_only  = (
    all_patients_df["DEATH_DATE_OBS"].notna() &
    all_patients_df["DEATH_DATE_ENC"].isna()  &
    all_patients_df["DEATH_DATE_PTS"].isna()
).sum()
pts_only  = (
    all_patients_df["DEATH_DATE_PTS"].notna() &
    all_patients_df["DEATH_DATE_ENC"].isna()  &
    all_patients_df["DEATH_DATE_OBS"].isna()
).sum()
all_three = (
    all_patients_df["DEATH_DATE_ENC"].notna() &
    all_patients_df["DEATH_DATE_OBS"].notna() &
    all_patients_df["DEATH_DATE_PTS"].notna()
).sum()

print(f"  All 3 sources agree   : {all_three:,}")
print(f"  Encounters only       : {enc_only:,}")
print(f"  Observations only     : {obs_only:,}")
print(f"  Patients.csv only     : {pts_only:,}")
print(f"  TOTAL deceased        : {all_patients_df['DIED'].sum():,} "
      f"({100*all_patients_df['DIED'].mean():.1f}%)")

# ============================================================
# STEP 7: Summary
# ============================================================

print("\n" + "="*55)
print("CELL 4 COMPLETE — OBSERVATION PERIODS")
print("="*55)
print(f"  Total patients          : {len(all_patients_df):,}")
print(f"  OBS_START range         : "
      f"{all_patients_df['OBS_START'].min().date()} → "
      f"{all_patients_df['OBS_START'].max().date()}")
print(f"  OBS_END range           : "
      f"{all_patients_df['OBS_END'].min().date()} → "
      f"{all_patients_df['OBS_END'].max().date()}")
print(f"  Duration mean / median  : "
      f"{all_patients_df['OBS_DURATION_YEARS'].mean():.1f} / "
      f"{all_patients_df['OBS_DURATION_YEARS'].median():.1f} yrs")
print(f"  Deceased                : {all_patients_df['DIED'].sum():,} "
      f"({100*all_patients_df['DIED'].mean():.1f}%)")
print("="*55)
print("  Ready — run Cell 5 to identify disease patients")
print("="*55)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 5: Disease Patient Identification & First Treatment
# ============================================================
# TWO-STEP APPROACH:
#
# Step 1 — WHO has the disease?
#   Identify disease patients using conditions.csv ONLY.
#   This is the canonical diagnostic source in Synthea.
#   Encounters/procedures are NOT used for identification
#   because REASONDESCRIPTION in those files captures visits
#   related to a condition, not a diagnosis — using them
#   would massively inflate the patient count.
#
# Step 2 — WHEN was their first evidence?
#   Once patients are confirmed, find the earliest date
#   of any disease-related record across ALL source files
#   for those confirmed patients only.
#
# Then finds first treatment date using TREATMENT_DESCRIPTIONS
# from Cell 3.
#
# Output variables used by downstream cells:
#   disease_patients_df   disease patients with first dx date,
#                         first treatment, and all obs/death info
# ============================================================

# ============================================================
# STEP 1: Identify disease patients from conditions.csv ONLY
# ============================================================

print(f"Step 1: Identifying {DISEASE_NAME} patients from conditions.csv...")

conditions = pd.read_csv(
    os.path.join(BASE_PATH, "conditions.csv"), low_memory=False
)
conditions["START"] = to_naive(conditions["START"])

# Match on DESCRIPTION
matched_by_desc = pd.DataFrame()
if "DESCRIPTION" in conditions.columns:
    mask = conditions["DESCRIPTION"].isin(CONDITION_DESCRIPTIONS)
    matched_by_desc = conditions[mask].copy()

# Match on CODE
matched_by_code = pd.DataFrame()
if "CODE" in conditions.columns and len(CONDITION_CODE_LIST) > 0:
    mask = conditions["CODE"].astype(str).isin(
        [str(c) for c in CONDITION_CODE_LIST]
    )
    matched_by_code = conditions[mask].copy()

# Combine and deduplicate
matched_conditions = pd.concat(
    [matched_by_desc, matched_by_code]
).drop_duplicates()

# Apply exclusions
if EXCLUDE_KEYWORDS:
    excl_pattern = "|".join(EXCLUDE_KEYWORDS)
    excl_mask = pd.Series(False, index=matched_conditions.index)
    for col in ["DESCRIPTION", "CODE"]:
        if col in matched_conditions.columns:
            excl_mask = excl_mask | \
                matched_conditions[col].astype(str).str.contains(
                    excl_pattern, case=False, na=False
                )
    n_before = len(matched_conditions)
    matched_conditions = matched_conditions[~excl_mask]
    n_excluded = n_before - len(matched_conditions)
    if n_excluded > 0:
        print(f"  Excluded {n_excluded:,} rows matching EXCLUDE_KEYWORDS")

# Confirmed disease patients set
confirmed_patients = set(matched_conditions["PATIENT"].unique())

print(f"  {DISEASE_NAME} patients confirmed : {len(confirmed_patients):,}")
print(f"  Matched descriptions:")
for desc, count in matched_conditions["DESCRIPTION"].value_counts().items():
    print(f"    {count:>5}  {desc}")

del matched_by_desc, matched_by_code

# ============================================================
# STEP 2: Find earliest disease evidence date
#         across ALL source files — confirmed patients only
# ============================================================

print(f"\nStep 2: Finding earliest {DISEASE_NAME} evidence date...")
print("  (searching across all source files for confirmed patients)")
print("-" * 55)

all_disease_dates = []

for source, info in CONDITION_SOURCE_MAP.items():
    cfg      = info["config"]
    filepath = os.path.join(BASE_PATH, cfg["csv"])

    if not os.path.exists(filepath):
        print(f"  MISSING : {cfg['csv']} — skipping")
        continue

    df       = pd.read_csv(filepath, low_memory=False)
    date_col = cfg["date_col"]
    pat_col  = cfg["patient_col"]

    if date_col not in df.columns or pat_col not in df.columns:
        print(f"  SKIP    : {cfg['csv']} — missing expected columns")
        continue

    df[date_col] = to_naive(df[date_col])

    # CRITICAL: filter to confirmed patients FIRST
    # before any description matching
    df = df[df[pat_col].isin(confirmed_patients)]

    if len(df) == 0:
        print(f"  EMPTY   : {cfg['csv']} — no confirmed patients found")
        continue

    matched = pd.DataFrame()

    # Match on DESCRIPTION
    if "DESCRIPTION" in cfg["match_cols"] and "DESCRIPTION" in df.columns:
        mask = df["DESCRIPTION"].isin(CONDITION_DESCRIPTIONS)
        matched = pd.concat([matched, df[mask]])

    # Match on CODE
    if "CODE" in df.columns and len(CONDITION_CODE_LIST) > 0:
        mask = df["CODE"].astype(str).isin(
            [str(c) for c in CONDITION_CODE_LIST]
        )
        matched = pd.concat([matched, df[mask]])

    # Match on REASONDESCRIPTION
    if "REASONDESCRIPTION" in cfg["match_cols"] and \
       "REASONDESCRIPTION" in df.columns:
        mask = df["REASONDESCRIPTION"].isin(CONDITION_DESCRIPTIONS)
        matched = pd.concat([matched, df[mask]])

    matched = matched.drop_duplicates()

    if len(matched) == 0:
        print(f"  NONE    : {cfg['csv']} — no matching records")
        continue

    rows = pd.DataFrame({
        "PATIENT": matched[pat_col].values,
        "DX_DATE": matched[date_col].values,
        "SOURCE":  source,
    })
    all_disease_dates.append(rows)

    print(f"  OK  {cfg['csv']:<30} "
          f"{len(matched):>6,} rows | "
          f"{matched[pat_col].nunique():,} patients")
    del df, matched

print("-" * 55)

# ============================================================
# STEP 3: Compute first diagnosis date per patient
# ============================================================

print("\nComputing first diagnosis date per patient...")

all_dx = pd.concat(all_disease_dates, ignore_index=True)
all_dx["DX_DATE"] = to_naive(all_dx["DX_DATE"])

first_dx = (
    all_dx.sort_values("DX_DATE")
    .groupby("PATIENT")
    .agg(
        FIRST_DX_DATE   = ("DX_DATE", "first"),
        FIRST_DX_SOURCE = ("SOURCE",  "first"),
    )
    .reset_index()
)

print(f"  Patients with first dx date : {len(first_dx):,}")
print(f"  First diagnosis source breakdown:")
for src, count in first_dx["FIRST_DX_SOURCE"].value_counts().items():
    print(f"    {src:<20} : {count:,} patients")

# ============================================================
# STEP 4: Merge with all_patients_df
# ============================================================

print("\nMerging with observation periods and death info...")

disease_patients_df = all_patients_df.merge(
    first_dx, on="PATIENT", how="inner"
)
disease_patients_df["FIRST_DX_DATE"] = to_naive(
    disease_patients_df["FIRST_DX_DATE"]
)

print(f"  {DISEASE_NAME} patients with obs periods : {len(disease_patients_df):,}")
print(f"  Deceased                              : "
      f"{disease_patients_df['DIED'].sum():,} "
      f"({100*disease_patients_df['DIED'].mean():.1f}%)")

# Age at diagnosis
disease_patients_df["AGE_AT_DX"] = (
    (disease_patients_df["FIRST_DX_DATE"] -
     disease_patients_df["BIRTHDATE"]).dt.days / 365.25
).round(1)

# ============================================================
# STEP 5: Find first treatment date
# ============================================================

print(f"\nFinding first treatment ({DISEASE_NAME})...")

if len(TREATMENT_DESCRIPTIONS) == 0:
    print("  WARNING: No treatment descriptions — skipping")
    disease_patients_df["FIRST_TREATMENT_DATE"] = pd.NaT
    disease_patients_df["FIRST_TREATMENT_DRUG"] = None
else:
    meds = pd.read_csv(
        os.path.join(BASE_PATH, "medications.csv"), low_memory=False
    )
    meds["START"] = to_naive(meds["START"])

    # Filter to confirmed disease patients first
    meds = meds[meds["PATIENT"].isin(confirmed_patients)]

    # Match on DESCRIPTION
    treatment_rows = meds[
        meds["DESCRIPTION"].isin(TREATMENT_DESCRIPTIONS)
    ].copy()

    # Also match on CODE
    if len(TREATMENT_CODE_LIST) > 0:
        code_rows = meds[
            meds["CODE"].astype(str).isin(
                [str(c) for c in TREATMENT_CODE_LIST]
            )
        ].copy()
        treatment_rows = pd.concat(
            [treatment_rows, code_rows]
        ).drop_duplicates()

    first_tx = (
        treatment_rows.sort_values("START")
        .groupby("PATIENT")
        .agg(
            FIRST_TREATMENT_DATE = ("START",       "first"),
            FIRST_TREATMENT_DRUG = ("DESCRIPTION", "first"),
        )
        .reset_index()
    )

    disease_patients_df = disease_patients_df.merge(
        first_tx, on="PATIENT", how="left"
    )
    disease_patients_df["FIRST_TREATMENT_DATE"] = to_naive(
        disease_patients_df["FIRST_TREATMENT_DATE"]
    )

    n_with    = disease_patients_df["FIRST_TREATMENT_DATE"].notna().sum()
    n_without = disease_patients_df["FIRST_TREATMENT_DATE"].isna().sum()

    print(f"  With treatment    : {n_with:,} ({100*n_with/len(disease_patients_df):.1f}%)")
    print(f"  Without treatment : {n_without:,} ({100*n_without/len(disease_patients_df):.1f}%)")
    print(f"  Drug breakdown:")
    for drug, count in treatment_rows["DESCRIPTION"].value_counts().items():
        print(f"    {count:>5}  {drug}")

    del meds, treatment_rows, first_tx

# ============================================================
# STEP 6: Summary
# ============================================================

print("\n" + "="*55)
print(f"CELL 5 COMPLETE — {DISEASE_NAME} PATIENTS IDENTIFIED")
print("="*55)
print(f"  Total {DISEASE_NAME} patients      : {len(disease_patients_df):,}")
print(f"  First Dx range          : "
      f"{disease_patients_df['FIRST_DX_DATE'].min().date()} → "
      f"{disease_patients_df['FIRST_DX_DATE'].max().date()}")
print(f"  Age at Dx mean / median : "
      f"{disease_patients_df['AGE_AT_DX'].mean():.1f} / "
      f"{disease_patients_df['AGE_AT_DX'].median():.1f} yrs")
print(f"  Deceased                : "
      f"{disease_patients_df['DIED'].sum():,} "
      f"({100*disease_patients_df['DIED'].mean():.1f}%)")
print(f"  With treatment          : "
      f"{disease_patients_df['FIRST_TREATMENT_DATE'].notna().sum():,} "
      f"({100*disease_patients_df['FIRST_TREATMENT_DATE'].notna().mean():.1f}%)")
print("="*55)
print("  Ready — run Cell 6 for index date assignment")
print("="*55)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 6: Index Date Assignment & Eligibility
# ============================================================
# Computes four index dates per patient and applies strict
# eligibility criteria for each.
#
# ELIGIBILITY RULE (all survival analyses):
#   DATASET_END_DATE - index_date >= FOLLOWUP_YEARS
#
# NOTE: We use DATASET_END_DATE (not patient OBS_END) to
# avoid survivorship bias — see Cell 2 for full explanation.
#
# RANDOM DATE WINDOW:
#   [FIRST_DX_DATE, min(FINAL_DEATH_DATE, DATASET_END_DATE)
#                   - FOLLOWUP_YEARS]
#   Capped at death date so the random index date never
#   falls after the patient has already died.
#
# Output variables used by downstream cells:
#   eligibility_df    all disease patients with index dates,
#                     lookback/followup years, eligibility flags
# ============================================================

import numpy as np

print(f"Computing index dates and eligibility for {DISEASE_NAME}...")
print(f"  DATASET_END_DATE : {DATASET_END_DATE.date()}")
print(f"  LOOKBACK_YEARS   : {LOOKBACK_YEARS}")
print(f"  FOLLOWUP_YEARS   : {FOLLOWUP_YEARS}")
print(f"  RANDOM_SEED      : {RANDOM_SEED}")
print("-" * 55)

# Work on a clean copy
master = disease_patients_df.copy()

# Ensure all date columns are tz-naive datetime
for col in ["OBS_START", "OBS_END", "FIRST_DX_DATE",
            "FIRST_TREATMENT_DATE", "FINAL_DEATH_DATE", "BIRTHDATE"]:
    if col in master.columns:
        master[col] = to_naive(master[col])

lookback_days  = LOOKBACK_YEARS  * 365.25
followup_days  = FOLLOWUP_YEARS  * 365.25

# ============================================================
# INDEX DATE 1 — First Diagnosis
# ============================================================

print("\nIndex Date 1 — First Diagnosis...")

master["INDEX_FIRST_DX"] = master["FIRST_DX_DATE"]

# Lookback: years of data available before index date
master["LOOKBACK_YRS_DX"] = (
    (master["INDEX_FIRST_DX"] - master["OBS_START"]).dt.days / 365.25
).round(2)

# Follow-up: years of dataset available after index date
master["FOLLOWUP_YRS_DX"] = (
    (DATASET_END_DATE - master["INDEX_FIRST_DX"]).dt.days / 365.25
).round(2)

# Eligible: index date exists + lookback + followup met
# Also: patient must be alive ON the index date
master["ELIGIBLE_FIRST_DX"] = (
    master["INDEX_FIRST_DX"].notna() &
    (master["LOOKBACK_YRS_DX"] >= LOOKBACK_YEARS) &
    (master["FOLLOWUP_YRS_DX"] >= FOLLOWUP_YEARS) &
    (
        master["FINAL_DEATH_DATE"].isna() |
        (master["FINAL_DEATH_DATE"] >= master["INDEX_FIRST_DX"])
    )
)

n = master["ELIGIBLE_FIRST_DX"].sum()
print(f"  Eligible : {n:,} / {len(master):,} "
      f"({100*n/len(master):.1f}%)")

# ============================================================
# INDEX DATE 2 — First Treatment
# ============================================================

print("Index Date 2 — First Treatment...")

master["INDEX_FIRST_TX"] = master["FIRST_TREATMENT_DATE"]

master["LOOKBACK_YRS_TX"] = (
    (master["INDEX_FIRST_TX"] - master["OBS_START"]).dt.days / 365.25
).round(2)

master["FOLLOWUP_YRS_TX"] = (
    (DATASET_END_DATE - master["INDEX_FIRST_TX"]).dt.days / 365.25
).round(2)

master["ELIGIBLE_FIRST_TX"] = (
    master["INDEX_FIRST_TX"].notna() &
    (master["LOOKBACK_YRS_TX"] >= LOOKBACK_YEARS) &
    (master["FOLLOWUP_YRS_TX"] >= FOLLOWUP_YEARS) &
    (
        master["FINAL_DEATH_DATE"].isna() |
        (master["FINAL_DEATH_DATE"] >= master["INDEX_FIRST_TX"])
    )
)

n = master["ELIGIBLE_FIRST_TX"].sum()
print(f"  Eligible : {n:,} / {len(master):,} "
      f"({100*n/len(master):.1f}%)")

# ============================================================
# INDEX DATE 3 — Random Date
#
# Sampling window per patient:
#   window_start = FIRST_DX_DATE
#   window_end   = min(FINAL_DEATH_DATE, DATASET_END_DATE)
#                  - FOLLOWUP_YEARS
#
# Capped at death date so the random index date never
# falls after the patient has already died.
# If window_end <= window_start the patient is ineligible.
# ============================================================

print("Index Date 3 — Random Date (seed={})...".format(RANDOM_SEED))

rng = np.random.default_rng(RANDOM_SEED)

random_dates = []
for _, row in master.iterrows():
    window_start = row["FIRST_DX_DATE"]

    # Upper bound: min(death date, dataset end) - followup
    if pd.notna(row["FINAL_DEATH_DATE"]):
        upper_anchor = min(row["FINAL_DEATH_DATE"], DATASET_END_DATE)
    else:
        upper_anchor = DATASET_END_DATE

    window_end = upper_anchor - pd.Timedelta(days=followup_days)

    if pd.isna(window_start) or pd.isna(window_end):
        random_dates.append(pd.NaT)
        continue

    window_days = (window_end - window_start).days
    if window_days <= 0:
        random_dates.append(pd.NaT)
    else:
        offset = int(rng.integers(0, window_days))
        random_dates.append(window_start + pd.Timedelta(days=offset))

master["INDEX_RANDOM"] = pd.to_datetime(random_dates)
master["RANDOM_SEED"]  = RANDOM_SEED

master["LOOKBACK_YRS_RND"] = (
    (master["INDEX_RANDOM"] - master["OBS_START"]).dt.days / 365.25
).round(2)

master["FOLLOWUP_YRS_RND"] = (
    (DATASET_END_DATE - master["INDEX_RANDOM"]).dt.days / 365.25
).round(2)

master["ELIGIBLE_RANDOM"] = (
    master["INDEX_RANDOM"].notna() &
    (master["LOOKBACK_YRS_RND"] >= LOOKBACK_YEARS) &
    (master["FOLLOWUP_YRS_RND"] >= FOLLOWUP_YEARS) &
    (
        master["FINAL_DEATH_DATE"].isna() |
        (master["FINAL_DEATH_DATE"] >= master["INDEX_RANDOM"])
    )
)

n = master["ELIGIBLE_RANDOM"].sum()
print(f"  Eligible : {n:,} / {len(master):,} "
      f"({100*n/len(master):.1f}%)")

# ============================================================
# INDEX DATE 4 — Latest (cross-sectional)
#
# Index date = OBS_END (last recorded clinical event)
# No follow-up requirement — cross-sectional only.
# Only requires minimum lookback.
# ============================================================

print("Index Date 4 — Latest (cross-sectional)...")

master["INDEX_LATEST"]      = master["OBS_END"]
master["LOOKBACK_YRS_LATEST"] = (
    (master["OBS_END"] - master["OBS_START"]).dt.days / 365.25
).round(2)

master["ELIGIBLE_LATEST"] = (
    master["LOOKBACK_YRS_LATEST"] >= LOOKBACK_YEARS
)

n = master["ELIGIBLE_LATEST"].sum()
print(f"  Eligible : {n:,} / {len(master):,} "
      f"({100*n/len(master):.1f}%)")

# ============================================================
# Age at diagnosis + age at death
# ============================================================

master["AGE_AT_DX"] = (
    (master["FIRST_DX_DATE"] - master["BIRTHDATE"]).dt.days / 365.25
).round(1)

master["AGE_AT_DEATH"] = np.where(
    master["DIED"],
    ((master["FINAL_DEATH_DATE"] - master["BIRTHDATE"]).dt.days / 365.25
     ).round(1),
    np.nan
)

# ============================================================
# Strip timezone from all datetime columns
# ============================================================

for col in master.select_dtypes(include=["datetimetz"]).columns:
    master[col] = master[col].dt.tz_localize(None)

# ============================================================
# Final column selection and rename
# ============================================================

eligibility_df = master[[
    "PATIENT",
    "GENDER", "RACE", "BIRTHDATE",
    "OBS_START", "OBS_END", "OBS_DURATION_YEARS",

    # First Diagnosis
    "FIRST_DX_DATE", "FIRST_DX_SOURCE", "AGE_AT_DX",
    "INDEX_FIRST_DX",
    "LOOKBACK_YRS_DX", "FOLLOWUP_YRS_DX", "ELIGIBLE_FIRST_DX",

    # First Treatment
    "FIRST_TREATMENT_DATE", "FIRST_TREATMENT_DRUG",
    "INDEX_FIRST_TX",
    "LOOKBACK_YRS_TX", "FOLLOWUP_YRS_TX", "ELIGIBLE_FIRST_TX",

    # Random Date
    "INDEX_RANDOM", "RANDOM_SEED",
    "LOOKBACK_YRS_RND", "FOLLOWUP_YRS_RND", "ELIGIBLE_RANDOM",

    # Latest
    "INDEX_LATEST",
    "LOOKBACK_YRS_LATEST", "ELIGIBLE_LATEST",

    # Outcome
    "DIED", "FINAL_DEATH_DATE", "FINAL_CAUSE_OF_DEATH",
    "AGE_AT_DEATH",
]].copy().rename(columns={
    "PATIENT":              "PATIENT_ID",
    "FINAL_DEATH_DATE":     "DEATH_DATE",
    "FINAL_CAUSE_OF_DEATH": "CAUSE_OF_DEATH",
})

# Format date columns to date only (no time component)
date_cols = [
    "BIRTHDATE", "OBS_START", "OBS_END",
    "FIRST_DX_DATE", "FIRST_TREATMENT_DATE",
    "INDEX_FIRST_DX", "INDEX_FIRST_TX",
    "INDEX_RANDOM", "INDEX_LATEST", "DEATH_DATE",
]
for col in date_cols:
    if col in eligibility_df.columns:
        eligibility_df[col] = pd.to_datetime(
            eligibility_df[col], errors="coerce"
        ).dt.date

# Sort by first diagnosis date
eligibility_df = eligibility_df.sort_values(
    "FIRST_DX_DATE"
).reset_index(drop=True)

# ============================================================
# Summary
# ============================================================

n_all    = len(eligibility_df)
n_dx     = eligibility_df["ELIGIBLE_FIRST_DX"].sum()
n_tx     = eligibility_df["ELIGIBLE_FIRST_TX"].sum()
n_rnd    = eligibility_df["ELIGIBLE_RANDOM"].sum()
n_latest = eligibility_df["ELIGIBLE_LATEST"].sum()
n_all3   = (
    eligibility_df["ELIGIBLE_FIRST_DX"] &
    eligibility_df["ELIGIBLE_FIRST_TX"] &
    eligibility_df["ELIGIBLE_RANDOM"]
).sum()
n_all4   = (
    eligibility_df["ELIGIBLE_FIRST_DX"] &
    eligibility_df["ELIGIBLE_FIRST_TX"] &
    eligibility_df["ELIGIBLE_RANDOM"] &
    eligibility_df["ELIGIBLE_LATEST"]
).sum()

print("\n" + "="*55)
print(f"CELL 6 COMPLETE — INDEX DATE ELIGIBILITY")
print("="*55)
print(f"  Total {DISEASE_NAME} patients      : {n_all:,}")
print(f"  Eligible — First Diagnosis    : {n_dx:,} ({100*n_dx/n_all:.1f}%)")
print(f"  Eligible — First Treatment    : {n_tx:,} ({100*n_tx/n_all:.1f}%)")
print(f"  Eligible — Random Date        : {n_rnd:,} ({100*n_rnd/n_all:.1f}%)")
print(f"  Eligible — Latest             : {n_latest:,} ({100*n_latest/n_all:.1f}%)")
print(f"  Eligible — ALL 3 survival     : {n_all3:,} ({100*n_all3/n_all:.1f}%)")
print(f"  Eligible — ALL 4              : {n_all4:,} ({100*n_all4/n_all:.1f}%)")
print(f"  Deceased                      : "
      f"{eligibility_df['DIED'].sum():,} "
      f"({100*eligibility_df['DIED'].mean():.1f}%)")
print("="*55)
print("  Ready — run Cell 7 to filter final cohort")
print("="*55)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 6b: Eligibility Audit
# ============================================================
# Explains WHY each ineligible patient fails eligibility
# for each index date. Run after Cell 6, before Cell 7.
#
# Ineligibility reasons:
#   A — No index date (missing first dx / treatment)
#   B — Insufficient lookback (< 5 yrs before index date)
#   C — Insufficient dataset coverage (< 5 yrs after index
#       date in the dataset) — note: this includes patients
#       who died, since dying does NOT disqualify a patient
#   D — Index date falls after death date (data quality)
# ============================================================

print(f"Eligibility audit for {DISEASE_NAME}")
print(f"Total patients: {len(eligibility_df):,}")
print("=" * 60)

def audit_eligibility(df, index_col, lookback_col, followup_col,
                      elig_col, label):
    """
    Break down ineligible patients by reason.
    Returns a summary dict.
    """
    inelig = df[~df[elig_col]].copy()
    n_total  = len(df)
    n_elig   = df[elig_col].sum()
    n_inelig = len(inelig)

    if n_inelig == 0:
        print(f"\n  {label}: ALL {n_total} patients eligible")
        return

    print(f"\n  {label}")
    print(f"  {'─'*50}")
    print(f"  Eligible   : {n_elig:,} / {n_total:,} "
          f"({100*n_elig/n_total:.1f}%)")
    print(f"  Ineligible : {n_inelig:,} / {n_total:,} "
          f"({100*n_inelig/n_total:.1f}%)")

    # Reason A — missing index date
    reason_a = inelig[index_col].isna()

    # Reason B — insufficient lookback
    reason_b = (
        inelig[index_col].notna() &
        (inelig[lookback_col] < LOOKBACK_YEARS)
    )

    # Reason C — insufficient dataset coverage after index date
    # (includes patients who died AND patients too recent)
    reason_c = (
        inelig[index_col].notna() &
        (inelig[lookback_col] >= LOOKBACK_YEARS) &
        (inelig[followup_col] < FOLLOWUP_YEARS)
    )

    # Reason D — index date after death (data quality flag)
    if "DEATH_DATE" in inelig.columns:
        reason_d = (
            inelig[index_col].notna() &
            inelig["DEATH_DATE"].notna() &
            (pd.to_datetime(inelig[index_col]) >
             pd.to_datetime(inelig["DEATH_DATE"]))
        )
    else:
        reason_d = pd.Series(False, index=inelig.index)

    print(f"\n  Breakdown of ineligible patients:")
    print(f"    A. No index date recorded      : {reason_a.sum():>4}")
    print(f"    B. Insufficient lookback        : {reason_b.sum():>4}  "
          f"(< {LOOKBACK_YEARS} yrs before index date)")
    print(f"    C. Insufficient dataset coverage: {reason_c.sum():>4}  "
          f"(< {FOLLOWUP_YEARS} yrs after index date in dataset)")
    print(f"    D. Index date after death       : {reason_d.sum():>4}  "
          f"(data quality flag)")

    # Further split reason C into:
    #   C1 — died within follow-up window (short dataset coverage
    #        because patient died AND dataset ended close to death)
    #   C2 — index date too recent (dataset simply doesn't extend
    #        5 years beyond the index date)
    reason_c_rows = inelig[reason_c]
    if len(reason_c_rows) > 0 and "DIED" in reason_c_rows.columns:
        c1_died = reason_c & inelig["DIED"]
        c2_alive = reason_c & ~inelig["DIED"]
        print(f"\n    Reason C breakdown:")
        print(f"      C1. Deceased patients         : {c1_died.sum():>4}  "
              f"(died — dataset coverage < {FOLLOWUP_YEARS} yrs after index)")
        print(f"      C2. Alive but index too recent: {c2_alive.sum():>4}  "
              f"(alive — dataset ends < {FOLLOWUP_YEARS} yrs after index)")

        # For C1 — show how long they survived after index date
        if c1_died.sum() > 0:
            c1_rows = inelig[c1_died].copy()
            c1_rows["SURVIVAL_YRS"] = (
                (pd.to_datetime(c1_rows["DEATH_DATE"]) -
                 pd.to_datetime(c1_rows[index_col])).dt.days / 365.25
            )
            print(f"\n      C1 survival after index date:")
            print(f"        mean   = {c1_rows['SURVIVAL_YRS'].mean():.1f} yrs")
            print(f"        median = {c1_rows['SURVIVAL_YRS'].median():.1f} yrs")
            print(f"        range  = {c1_rows['SURVIVAL_YRS'].min():.1f} – "
                  f"{c1_rows['SURVIVAL_YRS'].max():.1f} yrs")
            print(f"      NOTE: These patients are correctly ELIGIBLE")
            print(f"            (they contribute time-to-event with EVENT=1)")
            print(f"            They are excluded here only because FOLLOWUP_YRS")
            print(f"            check uses DATASET_END_DATE, which may not extend")
            print(f"            5 yrs past their index date if they died recently.")

    print(f"\n    Lookback distribution (ineligible):")
    print(f"      mean   = {inelig[lookback_col].mean():.1f} yrs")
    print(f"      median = {inelig[lookback_col].median():.1f} yrs")
    print(f"      range  = {inelig[lookback_col].min():.1f} – "
          f"{inelig[lookback_col].max():.1f} yrs")
    print(f"\n    Follow-up distribution (ineligible):")
    print(f"      mean   = {inelig[followup_col].mean():.1f} yrs")
    print(f"      median = {inelig[followup_col].median():.1f} yrs")
    print(f"      range  = {inelig[followup_col].min():.1f} – "
          f"{inelig[followup_col].max():.1f} yrs")

# ============================================================
# Run audit for each index date
# ============================================================

audit_eligibility(
    eligibility_df,
    index_col    = "INDEX_FIRST_DX",
    lookback_col = "LOOKBACK_YRS_DX",
    followup_col = "FOLLOWUP_YRS_DX",
    elig_col     = "ELIGIBLE_FIRST_DX",
    label        = "FIRST DIAGNOSIS",
)

audit_eligibility(
    eligibility_df,
    index_col    = "INDEX_FIRST_TX",
    lookback_col = "LOOKBACK_YRS_TX",
    followup_col = "FOLLOWUP_YRS_TX",
    elig_col     = "ELIGIBLE_FIRST_TX",
    label        = "FIRST TREATMENT",
)

audit_eligibility(
    eligibility_df,
    index_col    = "INDEX_RANDOM",
    lookback_col = "LOOKBACK_YRS_RND",
    followup_col = "FOLLOWUP_YRS_RND",
    elig_col     = "ELIGIBLE_RANDOM",
    label        = "RANDOM DATE",
)

audit_eligibility(
    eligibility_df,
    index_col    = "INDEX_LATEST",
    lookback_col = "LOOKBACK_YRS_LATEST",
    followup_col = "LOOKBACK_YRS_LATEST",  # no followup for latest
    elig_col     = "ELIGIBLE_LATEST",
    label        = "LATEST (cross-sectional)",
)

# ============================================================
# Overall: patients ineligible for ALL survival analyses
# ============================================================

print("\n" + "="*60)
print("OVERALL INELIGIBILITY SUMMARY")
print("="*60)

inelig_any = ~(
    eligibility_df["ELIGIBLE_FIRST_DX"] |
    eligibility_df["ELIGIBLE_FIRST_TX"] |
    eligibility_df["ELIGIBLE_RANDOM"]
)

inelig_all3 = ~(
    eligibility_df["ELIGIBLE_FIRST_DX"] &
    eligibility_df["ELIGIBLE_FIRST_TX"] &
    eligibility_df["ELIGIBLE_RANDOM"]
)

print(f"\n  Ineligible for ALL 3 survival analyses : "
      f"{inelig_all3.sum():,}")
print(f"  Of these, deceased                     : "
      f"{eligibility_df.loc[inelig_all3, 'DIED'].sum():,}")
print(f"  Of these, alive                        : "
      f"{(~eligibility_df.loc[inelig_all3, 'DIED']).sum():,}")

print(f"\n  Diagnosis date distribution (ineligible for all 3):")
inelig_rows = eligibility_df[inelig_all3]
if len(inelig_rows) > 0:
    print(f"    First Dx range : "
          f"{pd.to_datetime(inelig_rows['FIRST_DX_DATE']).min().date()} → "
          f"{pd.to_datetime(inelig_rows['FIRST_DX_DATE']).max().date()}")
    print(f"    Age at Dx mean : "
          f"{inelig_rows['AGE_AT_DX'].mean():.1f} yrs")

print("\n" + "="*60)
print("AUDIT COMPLETE")
print("Review the above before running Cell 7 (final filter)")
print("="*60)

In [ ]:
# ============================================================
# MOSAIC — Pharmacoepidemiology Pipeline
# CELL 7: Final Eligible Cohort & Save
# ============================================================
# Filters eligibility_df to patients eligible for ALL
# four index date analyses simultaneously, prints a full
# cohort summary, and saves all outputs.
#
# Three output files:
#   {DISEASE}_all_patients.csv         all disease patients
#                                      with eligibility flags
#   {DISEASE}_eligible_cohort.csv      fully eligible cohort
#                                      only — input to MOSAIC
#                                      pipeline and analysis
#                                      notebook
#   {DISEASE}_exclusion_log.csv        one row per excluded
#                                      patient with reason
# ============================================================

print(f"Building final eligible cohort for {DISEASE_NAME}...")

# ============================================================
# STEP 1: Filter to fully eligible patients
# ============================================================

fully_eligible = eligibility_df[
    eligibility_df["ELIGIBLE_FIRST_DX"] &
    eligibility_df["ELIGIBLE_FIRST_TX"] &
    eligibility_df["ELIGIBLE_RANDOM"]   &
    eligibility_df["ELIGIBLE_LATEST"]
].copy().reset_index(drop=True)

n_total = len(eligibility_df)
n_elig  = len(fully_eligible)
n_excl  = n_total - n_elig

print(f"  Total {DISEASE_NAME} patients   : {n_total:,}")
print(f"  Fully eligible            : {n_elig:,} "
      f"({100*n_elig/n_total:.1f}%)")
print(f"  Excluded                  : {n_excl:,} "
      f"({100*n_excl/n_total:.1f}%)")

# ============================================================
# STEP 2: Build exclusion log
# ============================================================

excluded = eligibility_df[
    ~(
        eligibility_df["ELIGIBLE_FIRST_DX"] &
        eligibility_df["ELIGIBLE_FIRST_TX"] &
        eligibility_df["ELIGIBLE_RANDOM"]   &
        eligibility_df["ELIGIBLE_LATEST"]
    )
].copy()

def get_exclusion_reason(row):
    reasons = []
    if not row["ELIGIBLE_FIRST_DX"]:
        if pd.isna(row["INDEX_FIRST_DX"]):
            reasons.append("No first diagnosis date")
        elif row["LOOKBACK_YRS_DX"] < LOOKBACK_YEARS:
            reasons.append(
                f"First Dx: lookback {row['LOOKBACK_YRS_DX']:.1f} "
                f"< {LOOKBACK_YEARS} yrs"
            )
        elif row["FOLLOWUP_YRS_DX"] < FOLLOWUP_YEARS:
            reasons.append(
                f"First Dx: dataset coverage "
                f"{row['FOLLOWUP_YRS_DX']:.1f} < {FOLLOWUP_YEARS} yrs"
            )
    if not row["ELIGIBLE_FIRST_TX"]:
        if pd.isna(row["INDEX_FIRST_TX"]):
            reasons.append("No first treatment date")
        elif row["LOOKBACK_YRS_TX"] < LOOKBACK_YEARS:
            reasons.append(
                f"First Tx: lookback {row['LOOKBACK_YRS_TX']:.1f} "
                f"< {LOOKBACK_YEARS} yrs"
            )
        elif row["FOLLOWUP_YRS_TX"] < FOLLOWUP_YEARS:
            reasons.append(
                f"First Tx: dataset coverage "
                f"{row['FOLLOWUP_YRS_TX']:.1f} < {FOLLOWUP_YEARS} yrs"
            )
    if not row["ELIGIBLE_RANDOM"]:
        if pd.isna(row["INDEX_RANDOM"]):
            reasons.append("No valid random date window")
        elif row["LOOKBACK_YRS_RND"] < LOOKBACK_YEARS:
            reasons.append(
                f"Random: lookback {row['LOOKBACK_YRS_RND']:.1f} "
                f"< {LOOKBACK_YEARS} yrs"
            )
        elif row["FOLLOWUP_YRS_RND"] < FOLLOWUP_YEARS:
            reasons.append(
                f"Random: dataset coverage "
                f"{row['FOLLOWUP_YRS_RND']:.1f} < {FOLLOWUP_YEARS} yrs"
            )
    if not row["ELIGIBLE_LATEST"]:
        reasons.append(
            f"Latest: lookback {row['LOOKBACK_YRS_LATEST']:.1f} "
            f"< {LOOKBACK_YEARS} yrs"
        )
    return " | ".join(reasons) if reasons else "Unknown"

excluded["EXCLUSION_REASON"] = excluded.apply(
    get_exclusion_reason, axis=1
)

# ============================================================
# STEP 3: Cohort summary
# ============================================================

def pct(n, total):
    return f"{n:,} ({100*n/total:.1f}%)"

print("\n" + "="*55)
print(f"FINAL COHORT SUMMARY — {DISEASE_NAME} (N={n_elig:,})")
print("="*55)

print(f"\n  Sex:")
for g, n in fully_eligible["GENDER"].value_counts().items():
    print(f"    {g:<10} : {pct(n, n_elig)}")

print(f"\n  Race/Ethnicity:")
for r, n in fully_eligible["RACE"].value_counts().items():
    print(f"    {r:<30} : {pct(n, n_elig)}")

print(f"\n  Age at diagnosis:")
print(f"    mean   = {fully_eligible['AGE_AT_DX'].mean():.1f} yrs")
print(f"    median = {fully_eligible['AGE_AT_DX'].median():.1f} yrs")
print(f"    range  = {fully_eligible['AGE_AT_DX'].min():.1f} – "
      f"{fully_eligible['AGE_AT_DX'].max():.1f} yrs")

print(f"\n  Observation period:")
print(f"    OBS_START range : "
      f"{pd.to_datetime(fully_eligible['OBS_START']).min().date()} → "
      f"{pd.to_datetime(fully_eligible['OBS_START']).max().date()}")
print(f"    OBS_END range   : "
      f"{pd.to_datetime(fully_eligible['OBS_END']).min().date()} → "
      f"{pd.to_datetime(fully_eligible['OBS_END']).max().date()}")
print(f"    Duration mean   = "
      f"{fully_eligible['OBS_DURATION_YEARS'].mean():.1f} yrs")

print(f"\n  Index date ranges:")
print(f"    First Dx  : "
      f"{pd.to_datetime(fully_eligible['FIRST_DX_DATE']).min().date()} → "
      f"{pd.to_datetime(fully_eligible['FIRST_DX_DATE']).max().date()}")
print(f"    First Tx  : "
      f"{pd.to_datetime(fully_eligible['FIRST_TREATMENT_DATE']).min().date()} → "
      f"{pd.to_datetime(fully_eligible['FIRST_TREATMENT_DATE']).max().date()}")
print(f"    Random    : "
      f"{pd.to_datetime(fully_eligible['INDEX_RANDOM']).min().date()} → "
      f"{pd.to_datetime(fully_eligible['INDEX_RANDOM']).max().date()}")

print(f"\n  Mortality:")
n_dead  = fully_eligible["DIED"].sum()
n_alive = n_elig - n_dead
print(f"    Deceased : {pct(n_dead, n_elig)}")
print(f"    Alive    : {pct(n_alive, n_elig)}")

if n_dead > 0:
    dead = fully_eligible[fully_eligible["DIED"]]
    print(f"\n  Cause of death (top 10):")
    for cause, count in dead["CAUSE_OF_DEATH"].value_counts(
        dropna=False
    ).head(10).items():
        print(f"    {count:>4}  {cause}")

print(f"\n  First treatment drug breakdown:")
for drug, count in fully_eligible[
    "FIRST_TREATMENT_DRUG"
].value_counts(dropna=False).items():
    short = str(drug)[:55] + "..." if len(str(drug)) > 55 else str(drug)
    print(f"    {count:>4}  {short}")

print(f"\n  Follow-up available (eligible cohort):")
for col, label in [
    ("FOLLOWUP_YRS_DX",  "First Dx "),
    ("FOLLOWUP_YRS_TX",  "First Tx "),
    ("FOLLOWUP_YRS_RND", "Random   "),
]:
    vals = pd.to_numeric(fully_eligible[col], errors="coerce")
    print(f"    {label} : mean={vals.mean():.1f}  "
          f"median={vals.median():.1f}  "
          f"min={vals.min():.1f}  max={vals.max():.1f} yrs")

# ============================================================
# STEP 4: Save all outputs
# ============================================================

print("\nSaving outputs...")

# 1 — All disease patients with eligibility flags
out_all = os.path.join(
    OUTPUT_PATH, f"{DISEASE_NAME}_all_patients.csv"
)
eligibility_df.to_csv(out_all, index=False)
print(f"  All patients saved       → {out_all}")

# 2 — Fully eligible cohort
out_elig = os.path.join(
    OUTPUT_PATH, f"{DISEASE_NAME}_eligible_cohort.csv"
)
fully_eligible.to_csv(out_elig, index=False)
print(f"  Eligible cohort saved    → {out_elig}")

# 3 — Exclusion log
out_excl = os.path.join(
    OUTPUT_PATH, f"{DISEASE_NAME}_exclusion_log.csv"
)
excluded[[
    "PATIENT_ID", "GENDER", "RACE",
    "FIRST_DX_DATE", "AGE_AT_DX",
    "ELIGIBLE_FIRST_DX", "ELIGIBLE_FIRST_TX",
    "ELIGIBLE_RANDOM", "ELIGIBLE_LATEST",
    "LOOKBACK_YRS_DX", "FOLLOWUP_YRS_DX",
    "LOOKBACK_YRS_TX", "FOLLOWUP_YRS_TX",
    "DIED", "DEATH_DATE",
    "EXCLUSION_REASON",
]].to_csv(out_excl, index=False)
print(f"  Exclusion log saved      → {out_excl}")

print("\n" + "="*55)
print(f"CELL 7 COMPLETE")
print("="*55)
print(f"  {DISEASE_NAME} eligible cohort : {n_elig:,} patients")
print(f"  Files saved to : {OUTPUT_PATH}")
print(f"\n  Next steps:")
print(f"    → MOSAIC pipeline : run on {DISEASE_NAME}_eligible_cohort.csv")
print(f"    → Analysis NB     : load {DISEASE_NAME}_eligible_cohort.csv")
print(f"    → Review excluded : {DISEASE_NAME}_exclusion_log.csv")
print("="*55)